# G11C/G11D — GPU Sequence + Two-Tower Retrieval Batch (Colab)

Runs a **priority queue of models in one GPU session**, writing results **after each model** (disconnect-safe). Keeps the **exact G11/G11B eval** (5k-user leave-last-out, full 40k scoring, R@20/50 + NDCG@20 + bootstrap CI).

**Core queue (most valuable first):**
1. **SASRec CANONICAL — full-softmax over all 40k items** (impossible on CPU; the real sequence T3 attempt), d64/2blk/2head, 40 ep
2. SASRec-small d48 last-position, 20 ep (clean-pipeline sanity ≈ v1)
3. **TwoTower neural retrieval (G11D)** — industry-standard candidate retrieval; user tower = pooled history, item tower = item embedding; in-batch negatives

**Optional (set `G11C_OPTIONAL=0` to skip & save GPU):** SASRec v2 sampled-neg 30 ep diagnostic; GRU4Rec full-softmax 30 ep.

Est. ~25–40 min on a T4. Canonical SASRec saves first.

**Honesty rules:** no eval changes, no fabricated numbers. TwoTower (item-ID only) is a **neural collaborative retrieval baseline** — NOT a content cold-start or production-scale two-tower. No advanced-model-win claim unless the numbers earn it.

## Required uploads (7 files, from `g11c_upload/`)
`run_g11c_colab.py`, `g11_fullpos_arrays.npz`, `d3b_seq_meta.pkl`, `eval_sample.csv`, `domain_als_f64_floor.json`, `domain_sasrec_report.json`, `g11_sequence_tournament_metrics.json`

In [ ]:
import torch
print('torch', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — set Runtime ▸ Change runtime type ▸ GPU')

In [ ]:
# Upload the 7 files (run_g11c_colab.py + 6 data/comparator files). Click Choose Files, select all 7.
from google.colab import files
up = files.upload()
print('uploaded:', sorted(up.keys()))

In [ ]:
# Run the batch. Writes g11c_colab_metrics.json AFTER EACH model (disconnect-safe),
# plus g11c_training_log.csv and g11c_model_comparison.png.
# To run only the 3 core models (skip optional diagnostics):  !G11C_OPTIONAL=0 python run_g11c_colab.py . .
!python run_g11c_colab.py . .

In [ ]:
# Inspect results (safe any time, even mid-batch — file updates after each model)
import json
m = json.load(open('g11c_colab_metrics.json'))
print('device:', m['device'], '| gpu:', m['gpu'], '| elapsed_sec:', m['elapsed_sec'])
print('--- GPU runs ---')
for name, r in m['results_gpu'].items():
    if 'ERROR' in r: print(f'{name}: ERROR {r["ERROR"]}'); continue
    print(f"{name:46s} R@20 {r['R@20']['mean']:.4f}  R@50 {r['R@50']['mean']:.4f}  N@20 {r['N@20']['mean']:.4f}")
print('--- comparators ---')
for name, r in m['comparators'].items():
    print(f"{name:46s} R@20 {r['R@20']['mean']:.4f}")

In [ ]:
from google.colab import files
for f in ['g11c_colab_metrics.json', 'g11c_training_log.csv', 'g11c_model_comparison.png']:
    try: files.download(f)
    except Exception as e: print('skip', f, e)

## After the run
1. Copy `g11c_colab_metrics.json` + `g11c_training_log.csv` → `outputs/evidence/`, and `g11c_model_comparison.png` → `outputs/plots/`.
2. Tell Claude "G11C/D results are in" → it fills `docs/52` and sets the verdict.
3. Verdict per model family: **T3 gap closed** (≥ ALS f64 ~0.085) / **materially reduced** / **still open**.